# Issue: SRS images (Vsrs_g) full of NaN values after interpolation

# Plot set Vx_scan, Vy_scan to galvos and measured Vx_meas, Vy_meas

# Vx_meas disagrees with Vx_scan -> bad connection of Dev1/ai2 at the pin terminal

In [2]:
%cd ..
from experiment_control.srs_microscope.srs_microscope import *

c:\Users\euyehara\miniconda3\envs\instrumental\Lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


c:\Users\euyehara
Opening connnection to Keithley source meter
Opening connection to Agilent E3633A DC Power Supply
VOA initialized to 4.99 volt


In [ ]:
# dir_name = "Sample_Sample_GSdev1b_delayvolt13_2025-06-02-23-15-28_2025-06-03-00-30-28"
# fname = "GalvoScan_diamondCaroteneIPA_OEland1075.6-60mW_1mV300us_N300_1V_2025-06-03-00-41-36.h5"

dir_name = "Sample_CWdev1b_MitutoyoNIR20x_2025-10-05-21-44-29"
fname ="GalvoScan_diamond_OEland1075.4-81mW_100uV3ms_N300_srs_microscope_old_2025-10-05-22-10-40.h5"
fpath = os.path.join(data_dir, dir_name)

ds0 = load_data_from_file(fpath, fname)

nx = 300
ny=300
Vx_meas = ds0["Dev1"]["ai2"][1:] #discard 1st ai value (ai lags ao by 1 sample point)
Vy_meas = ds0["Dev1"]["ai3"][1:]
Vsrs = ds0["Dev1"]["ai0"][1:]

nx = ds0["Vsrs_g"].shape[0]
ny = ds0["Vsrs_g"].shape[1]

Vx = ds0["Vx"]
Vy = ds0["Vy"]

Vx_scan = np.tile(np.concatenate((Vx.m,Vx.m[::-1])),ny//2)*u.volt
Vy_scan = np.repeat(Vy.m,nx)*u.volt
Vx_g, Vy_g = np.meshgrid(Vx.m,Vy.m)

print(f"Vx_meas: {Vx_meas.shape}")
print(f"Vsrs: {Vsrs.shape}")
print(f"Vx_g: {Vx_g.shape}")
print(f"Vs_scan: {Vx_scan.shape}")
print(Vx.shape)

plt.figure()
plt.plot(Vx_scan, label='Vx_scan')
plt.plot(Vy_scan, label='Vy_scan')

plt.plot(Vx_meas, label='Vx meas')
plt.plot(Vy_meas, label='Vy_meas')
# plt.xlim((0,10000))

plt.legend()
plt.show()


In [27]:
Vx0, Vy0 = (0.21)*u.volt, (0.43)*u.volt 
Vx = 0.25*u.V
Vy = 0.1*u.V
move_spot(Vx, Vy)


Vx_target, Vy_target = (Vx+Vx0), (Vy+Vy0)
print(f'Vx_target: {Vx_target}')
print(f'Vy_target: {Vy_target}')

Vx, Vy = get_spot_pos()
print(f"Meas Vx pos: {Vx}")
print(f"Meas Vy pos: {Vy}")

print(f"Vx_meas: {ch_Vx_meas.read()}")
print(f"Vy_meas: {ch_Vy_meas.read()}")



Vx_target: 0.45999999999999996 volt
Vy_target: 0.53 volt
Meas Vx pos: 0.249265708373995 volt
Meas Vy pos: 0.1002357101475469 volt
Vx_meas: 0.4599627607157921 volt
Vy_meas: 0.5277068475200505 volt


# Debugging Vy_meas noise

In [ ]:
# dir_name = "Sample_Sample_GSdev1b_delayvolt13_2025-06-02-23-15-28_2025-06-03-00-30-28"
# fname = "GalvoScan_diamondCaroteneIPA_OEland1075.6-60mW_1mV300us_N300_1V_2025-06-03-00-41-36.h5"
%matplotlib widget
from scipy.signal import savgol_filter

# dir_name = "Sample_CWdev1b_MitutoyoNIR20x_2025-10-05-21-44-29"
# fname ="GalvoScan_diamond_OEland1075.4-81mW_100uV3ms_N300_srs_microscope_old_2025-10-05-22-10-40.h5"
dir_name = "Sample_GSdev1b_Nikon20x_delayvolt16_2025-11-02-18-06-04"
# fname ="GalvoScan_diamond_OEland1074.4-73mW_100uV300us_N150V_5tlia_2025-11-02-19-14-02.h5"
fname = "GalvoScan_test_Vy_meas_2025-11-06-17-49-33.h5"
fpath = os.path.join(data_dir, dir_name)

ds0 = load_data_from_file(fpath, fname)


Vx_meas = ds0["Dev1"]["ai2"][1:] #discard 1st ai value (ai lags ao by 1 sample point)
Vy_meas = ds0["Dev1"]["ai3"][1:]
Vsrs = ds0["Dev1"]["ai0"][1:]

nx = ds0["Vsrs_g"].shape[0]
ny = ds0["Vsrs_g"].shape[1]

Vx = ds0["Vx"]
Vy = ds0["Vy"]

Vx_scan = np.tile(np.concatenate((Vx.m,Vx.m[::-1])),ny//2)*u.volt
Vy_scan = np.repeat(Vy.m,nx)*u.volt
Vx_g, Vy_g = np.meshgrid(Vx.m,Vy.m)

print(f"Vx_meas: {Vx_meas.shape}")
print(f"Vsrs: {Vsrs.shape}")
print(f"Vx_g: {Vx_g.shape}")
print(f"Vs_scan: {Vx_scan.shape}")
print(Vx.shape)

plt.figure()
plt.plot(Vx_scan, '.-', label='Vx_scan')
plt.plot(Vy_scan, '.-', label='Vy_scan')

plt.plot(Vx_meas, '.-', label='Vx meas')
plt.plot(Vy_meas, '.-', label='Vy_meas')
# plt.xlim((0,2000))
# plt.ylim((0.36, 0.375))

plt.legend()
plt.show()

